# FADE · STAGE 3 — the test set, with no gold anywhere

The claim nothing else in this project can make. EDER needs gold to diagnose, so it
only ever runs on train. CTC is what carries the method across.

```
gen 1   model solves THIS test question       -> trace, answer
cpu     CTC reads (question, trace):
            p_wrong          is this trace wrong?
            predicted_type   which of the eight failures?
gen 2   if p_wrong >= 0.683: the cure, solved again, right now
        NO SCORING. gold is never opened.
```

Seeds and the exemplar pool stay exactly as the train run built them — they are the
frozen system, not part of the evaluation. Only the **questions** are new.

### What stage 2 established, and what it did not

| | stage 1 predicted | stage 2 delivered |
|---|---|---|
| detect AUROC | 0.776 | **0.770** |
| type accuracy | 0.598 | **0.590** |

Detection generalised almost exactly. So did typing *accuracy*. What did **not**
carry is the *benefit* of typing — on train the type came from the gold cascade and
was essentially always right; here it comes from CTC and is right 59% of the time.

Stage 2, on its own data:

```
cure matched the true type    15.0%  n=60
cure was mismatched           23.5%  n=34     <- wrong direction, p = 0.40
gated to the GENERIC cure     27.3%  n=33
```

Too few to conclude anything, and pointing the wrong way. **Cell 6 runs the ablation
that settles it** — the same flagged questions, generic cure for all of them.

### The number that survives review

`correct OR retry_correct` needs gold to pick the better of two answers. A deployed
FADE flagged the first answer and replaced it:

```
pass 1                              33.5%
UNION    correct OR retry_correct   40.0%   +6.5%   <- oracle, train-style
REPLACE  flagged -> retry answer    38.2%   +4.8%   <- the deployable claim, p=0.0013
```

On test there is no gold, so **REPLACE is the only thing that can be computed** — the
ambiguity disappears by construction.

### The control, free of charge

Computed from stage 2's own measured rates (recovery 20.5%, damage 63.6%):

| | TP | FP | REPLACE acc |
|---|---|---|---|
| pass-1 only | 0 | 0 | 33.5% |
| **random 138** | 92 | 46 | **30.8%** (−2.7%) |
| **CTC 138** | 127 | 11 | **38.2%** (+4.8%) |
| retry everything | 266 | 134 | 25.8% (−7.7%) |

An undirected second attempt *loses ground*. That is the claim: gold-free detection
makes a retry worth taking where an undirected one is harmful.

**Sidebar:** GPU T4/P100 · Internet on · Secrets `HF_TOKEN` and `GH_TOKEN`.
**Budget:** ~1319 test questions ≈ 6–7 GPU-h. Split across sessions; it resumes.

### 1 · Setup

In [ ]:
import os, sys, shutil, subprocess, json, datetime
!pip -q install -U "transformers>=4.40" accelerate sentence-transformers sympy scikit-learn datasets scipy joblib 2>/dev/null | tail -1
REPO='/kaggle/working/fade'; STORES='/kaggle/working/stores'; os.makedirs(STORES,exist_ok=True)
if os.path.exists(REPO): shutil.rmtree(REPO)
!git clone --depth 1 https://github.com/S2V3/fade.git {REPO}
sys.path.insert(0,REPO)
import importlib, config; importlib.reload(config)
VER=config.FADE_CODE_VERSION; assert VER=='audit-2', f'STALE CODE {VER!r}'
# ctc_train.py stays required only so an old v1 joblib can still be unpickled.
for fn in ('ctc_train.py','ctc_train2.py','ctc_data.py','ctc_features.py',
           'run_ctc_validate.py','cure_bank.py','negatives.py','autopush.py',
           'kaggle_run.py','symbolic_repair.py'):
    assert os.path.exists(f'{REPO}/{fn}'), f'{fn} missing -- push it to the repo'
# [D76] stage 2 must run the SAME cure as the typed arm.
import kaggle_run as _K
assert hasattr(_K,'apply_cure'), (
    'kaggle_run.py is the pre-D76 version -- stage 2 would run a WEAKER retry '
    '(no reroute, naming, prefill, resamples or repair). Push it first.')
from kaggle_secrets import UserSecretsClient
_us=UserSecretsClient(); HF=_us.get_secret('HF_TOKEN'); GH=_us.get_secret('GH_TOKEN')
os.environ['FADE_HF_TOKEN']=HF
MODEL='meta-llama/Llama-2-7b-chat-hf'; TAG=VER.replace('-','')
print('code',VER,'| apply_cure present | OK')

### 2 · Capability guard

`run_ctc_validate.py` must accept every flag this notebook passes, **before** a GPU
is touched. The generic notebook once lost a whole session to a missing `--store-dir`
that the version banner did not catch. `--flag-budget` and `--type-conf-gate` are new
in this revision, so an un-pushed script fails here rather than three hours in.

In [ ]:
h = subprocess.run([sys.executable,'run_ctc_validate.py','--help'], cwd=REPO,
                   capture_output=True, text=True)
need = ['--ctc','--pool-store','--store-dir','--retry-flagged','--no-score',
        '--retrieval','--flag-budget','--type-conf-gate','--negatives','--split']
missing = [k for k in need if k not in h.stdout]
print(h.stdout[:600] if h.stdout else h.stderr[:600])
if missing:
    raise SystemExit(f"\n  STOP: run_ctc_validate.py is missing {missing}. Push the "
                     "updated file to the repo and re-run from cell 1.")
print('\n  guard passed')

In [ ]:
# --- github sync (nothing conceptual) ----------------------------------
AUTH=f'https://{GH}@github.com/S2V3/fade.git'; BRANCH='results'
def sh(cmd):
    p=subprocess.Popen(cmd,shell=True,cwd=REPO,stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT,text=True,bufsize=1)
    for ln in p.stdout: print(ln,end='',flush=True)
    p.wait(); return p.returncode
def _rows(p):
    f=os.path.join(p,'results.jsonl')
    return sum(1 for _ in open(f)) if os.path.exists(f) else 0
def _ver(p):
    try: return json.load(open(os.path.join(p,'config_snapshot.json'))).get('FADE_CODE_VERSION')
    except Exception: return None
def _clone(w):
    shutil.rmtree(w,ignore_errors=True)
    if subprocess.run(['git','clone','--depth','1','--branch',BRANCH,AUTH,w],
                      capture_output=True,text=True).returncode:
        subprocess.run(['git','clone','--depth','1',AUTH,w],check=True,capture_output=True,text=True)
        subprocess.run(['git','checkout','-b',BRANCH],cwd=w,capture_output=True,text=True)
def fetch_any(prefix, dest, need_file=None):
    """Restore the newest remote dir whose name starts with `prefix`."""
    w=f'/kaggle/working/_rs_{abs(hash(prefix))%9999}'; c=[]
    if subprocess.run(['git','clone','--depth','1','--branch',BRANCH,AUTH,w],
                      capture_output=True,text=True).returncode==0 and os.path.isdir(f'{w}/results'):
        for d in sorted(os.listdir(f'{w}/results')):
            if d.startswith(prefix):
                full=f'{w}/results/{d}'
                if os.path.isdir(full) and (need_file is None or os.path.exists(f'{full}/{need_file}')):
                    c.append((d, full))
    if c:
        c.sort()
        shutil.rmtree(dest,ignore_errors=True); shutil.copytree(c[-1][1], dest)
        print(f'  restored {c[-1][0]} -> {dest}')
        return True
    print(f'  nothing remote matching {prefix}*')
    return False
def fetch_store(run,dest):
    w=f'/kaggle/working/_rs_{run}'; c=[]
    if subprocess.run(['git','clone','--depth','1','--branch',BRANCH,AUTH,w],
                      capture_output=True,text=True).returncode==0 and os.path.isdir(f'{w}/results'):
        for d in sorted(os.listdir(f'{w}/results')):
            if d.startswith(f'store_{run}_'):
                full=f'{w}/results/{d}'
                if os.path.isdir(full) and _ver(full)==VER and _rows(full)>0: c.append((_rows(full),d,full))
    c.sort(reverse=True)
    print(f'  [{run}] local {_rows(dest)} | best remote {c[0][0] if c else 0}')
    if c and c[0][0]>_rows(dest):
        shutil.rmtree(dest,ignore_errors=True); shutil.copytree(c[0][2],dest); print(f'  RESTORED {c[0][0]}')
    return _rows(dest)
def push_dir(src,name):
    if not os.path.isdir(src): return
    stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    w='/kaggle/working/_ck'; _clone(w)
    for dest in (f'{w}/results/{name}_{stamp}', f'{w}/results/{name}_latest'):
        shutil.rmtree(dest,ignore_errors=True); shutil.copytree(src,dest)
    for c in (['git','config','user.email','fade@kaggle'],['git','config','user.name','fade'],
              ['git','add','-A'],['git','commit','-m',f'{name}_{stamp}']):
        subprocess.run(c,cwd=w,capture_output=True,text=True)
    p=subprocess.run(['git','push','-u','origin',BRANCH],cwd=w,capture_output=True,text=True)
    if p.returncode:
        subprocess.run(['git','pull','--rebase','origin',BRANCH],cwd=w,capture_output=True,text=True)
        p=subprocess.run(['git','push','-u','origin',BRANCH],cwd=w,capture_output=True,text=True)
    print(f"  push {name}: {'ok' if p.returncode==0 else 'FAILED -- ' + p.stderr.strip()[:120]}")
print('sync ready')

### 3 · Get the CTC v2 model — retraining locally if the push failed

Stage 1 saves `ctc2.joblib`. If it is not on the branch, retraining costs ~3 minutes
of CPU and is deterministic (`random_state=0`), so it reproduces the identical model
rather than a different one.

`ctc_data.py` assembles the training set from the **latest typed run only**, and
`ctc_train2.py` evaluates with `GroupKFold` by question — the split that does not let
a question's pass-1 trace teach its own retry.

In [ ]:
TRAIN_RUN=f'typed_train_{TAG}'; TRAIN=f'{STORES}/store_{TRAIN_RUN}'
CTC_DIR=f'{STORES}/ctc2_{TAG}'
n=fetch_store(TRAIN_RUN, TRAIN)
assert n>0, 'no finished typed train store on the results branch'

CTC=f'{CTC_DIR}/ctc2.joblib'
got = fetch_any(f'ctc2_{TAG}', CTC_DIR, need_file='ctc2.joblib')
if not got or not os.path.exists(CTC):
    print('\n  no CTC v2 on the branch -- retraining (CPU, ~3 min, deterministic)')
    os.makedirs(CTC_DIR, exist_ok=True)
    rc = sh(f'python ctc_data.py --stores {TRAIN} --out {CTC_DIR}/ctc_dataset.jsonl')
    assert rc==0, 'ctc_data.py failed'
    rc = sh(f'python ctc_train2.py --data {CTC_DIR}/ctc_dataset.jsonl '
            f'--out-dir {CTC_DIR}')
    assert rc==0, 'ctc_train2.py failed'
    push_dir(CTC_DIR, f'ctc2_{TAG}')
assert os.path.exists(CTC), 'still no ctc2.joblib'

rep=json.load(open(f'{CTC_DIR}/ctc2_report.json'))
print(f"\n  n {rep['n']}  wrong {rep['n_wrong']}")
print(f"  detect AUROC {rep['detect']['auroc']:.3f}"
      f"   (expect ~0.78 here -- this stage sees pass-1 traces only)")
print(f"  type   acc   {rep['type']['accuracy']:.3f}"
      f"  majority {rep['type']['majority']:.3f}"
      f"   (expect ~0.60 here, same reason)")
assert rep['detect']['auroc']>=0.60, 'detector too weak to validate'
assert rep['type']['accuracy']>rep['type']['majority'], 'type head no better than majority'
print('  ready')

### 4 · Stage 3 — test split, `--no-score`

`ATTEMPT` works exactly as before: bump for a fresh run, leave alone to resume.

`--split test` draws the **questions** from the test split while seeds and pool stay
as the train run built them. `--no-score` means gold is never opened — the output
rows carry no `gold_answer`, no `correct`, no `true_type`. Verify that in cell 5
rather than trusting it.

GSM8K test is 1319 questions. At ~15 s/solve plus ~35% retried that is roughly 6–7
GPU-h, more than one Kaggle session. Run it, let the session end, re-run with the
same `ATTEMPT` — it picks up where it stopped.

In [ ]:
import autopush

# ---------------------------------------------------------------- the knob
ATTEMPT = 1          # bump for a FRESH run. leave alone to RESUME that run.
# --------------------------------------------------------------------------
N=1319; OFFSET=0; RETRY_FLAGGED=True; PUSH_EVERY_MIN=10
FLAG_BUDGET=0.40     # stage 2 measured best_budget_measured = 0.4
TYPE_GATE=0.50       # same as stage 2, so the two are comparable
RETRIEVAL='3stage'

RUN=f'ctctest_{TAG}_a{ATTEMPT}'; STORE=f'{STORES}/store_{RUN}'
if not os.path.exists(f'{STORE}/results.jsonl'):
    shutil.rmtree(STORE, ignore_errors=True)
fetch_store(RUN, STORE); before=_rows(STORE)
print(f'\n  {"FRESH START" if before==0 else f"RESUMING: {before}/{N} done"}  attempt {ATTEMPT}')

rc = 1
ap = autopush.start(STORE, RUN, GH, minutes=PUSH_EVERY_MIN)
try:
    rc = sh(f'python -u run_ctc_validate.py --ctc {CTC} '
            f'--pool-store {TRAIN} --store-dir {STORE} --n-problems {N} '
            f'--split test --eval-offset {OFFSET} --no-score '
            f'--flag-budget {FLAG_BUDGET} --type-conf-gate {TYPE_GATE} '
            f'{"--retry-flagged" if RETRY_FLAGGED else "--no-retry-flagged"} '
            f'--retrieval {RETRIEVAL} --model {MODEL} --show-every 25 '
            f'--secret-name HF_TOKEN')
finally:
    ap.stop()
print(f'\n  attempt {ATTEMPT}: {before} -> {_rows(STORE)} of {N}  (exit {rc})')
if _rows(STORE) < N:
    print(f'  INCOMPLETE -- re-run this cell with ATTEMPT={ATTEMPT} in a new session.')
else:
    assert rc==0, f'FAILED exit {rc}'
    print('  complete.')

### If the session dies anyway

Everything up to the last checkpoint is on the branch under `store_{RUN}_*`. Re-run
cells 1–4 **without touching `ATTEMPT`**: `fetch_store` restores the largest matching
store and the runner skips the questions already done.

To force a push at any moment, from a new cell:

```python
import autopush
autopush.AutoPush(STORE, RUN, GH).push_now()
```

To abandon an attempt and start clean, bump `ATTEMPT`. The old store stays on the
branch untouched, so nothing is lost and the two attempts can be compared.

### 5 · Score it — the ONLY place gold is opened, and only at the end

In [ ]:
# Gold was never in the loop. It is opened HERE, once, on the finished traces.
import kaggle_run as K2
K2.SPLIT_TAG='test'; K2.MODEL_ID=MODEL
test={p['question']:p for p in K2._load_split('test', None)}
rows=[json.loads(l) for l in open(f'{STORE}/results.jsonl') if l.strip()]
leak=[k for r in rows for k in ('gold_answer','correct','true_type') if k in r]
assert not leak, f'GOLD IN THE RUN OUTPUT: {set(leak)} -- the run was not gold-free'
print(f'  {len(rows)} rows, no gold in any of them. Scoring now.\n')

p1=fin=flag=tp=fp=rec=dmg=0
for r in rows:
    prob=test.get(r['question'])
    if prob is None: continue
    c1=K2.score_trace(r['question'],r['trace'],prob['answer'],prob['gold_answer'])[0].correct
    p1+=c1
    if r.get('retry_trace'):
        c2=K2.score_trace(r['question'],r['retry_trace'],prob['answer'],prob['gold_answer'])[0].correct
        flag+=1; tp+=(not c1); fp+=c1
        rec+=(not c1) and c2; dmg+=c1 and (not c2)
        fin+=c2                       # REPLACE: the rejected answer is gone
    else:
        fin+=c1
n=len(rows)
print(f'  pass 1                 {p1:>5}/{n} = {p1/n:>6.1%}')
print(f'  final (REPLACE)        {fin:>5}/{n} = {fin/n:>6.1%}   ({(fin-p1)/n:+.1%})')
print(f'\n  flagged {flag} ({flag/n:.1%})   TP {tp}  FP {fp}   precision {tp/max(flag,1):.1%}')
print(f'  recovered {rec}/{tp} = {rec/max(tp,1):.1%}   broken {dmg}/{fp} = {dmg/max(fp,1):.1%}')
try:
    from stats import mcnemar
    a_={r['question']:False for r in rows}; b_={r['question']:False for r in rows}
    for r in rows:
        prob=test.get(r['question'])
        if prob is None: continue
        c1=K2.score_trace(r['question'],r['trace'],prob['answer'],prob['gold_answer'])[0].correct
        a_[r['question']]=bool(c1)
        if r.get('retry_trace'):
            b_[r['question']]=bool(K2.score_trace(r['question'],r['retry_trace'],
                                    prob['answer'],prob['gold_answer'])[0].correct)
        else: b_[r['question']]=bool(c1)
    mc=mcnemar(a_,b_)
    print(f"  McNemar p = {mc['p']:.2e}  gained {mc['n01']}  lost {mc['n10']}"
          + ('  SIGNIFICANT' if mc['p']<0.05 else '  not significant'))
except Exception as e: print('  McNemar:',e)
print(f"\n  RANDOM control at the same budget, using THESE measured rates:")
R=rec/max(tp,1); D=dmg/max(fp,1); W=n-p1
rnd=(p1-flag*p1/n)+ (flag*W/n)*R + (flag*p1/n)*(1-D)
print(f"    random {flag}  -> {rnd:.0f}/{n} = {rnd/n:.1%}   ({(rnd-p1)/n:+.1%})")
print(f"    CTC    {flag}  -> {fin}/{n} = {fin/n:.1%}   ({(fin-p1)/n:+.1%})")
print(f"    CTC over random: {(fin-rnd)/n:+.1%}")

### 6 · The ablation that settles typing — ~50 GPU-min

Same detector, same flagged questions, **generic cure for every one of them**
(`--type-conf-gate 1.01` sends everything down the abstention path). The only thing
that differs from cell 4 is the cure, so the comparison is clean and paired.

Run it on the **stage-2** 400 (fast, and gold is available to score it), then compare
`final_accuracy` against the stage-2 run you already have.

- typed **>** generic → the type head earns its place; report the +.
- typed **≈** generic → report detection as the contribution and typing as a
  measured null. Still a paper: *the failure type is predictable, but type-matched
  treatment does not beat an untyped retry.*
- typed **<** generic → drop the gate to 1.01 in stage 3 and report the negative.

Any of the three is publishable. Not knowing which is not.

In [ ]:
ABL_ATTEMPT = 1
ABL_RUN=f'ctcabl_generic_{TAG}_a{ABL_ATTEMPT}'; ABL_STORE=f'{STORES}/store_{ABL_RUN}'
if not os.path.exists(f'{ABL_STORE}/results.jsonl'):
    shutil.rmtree(ABL_STORE, ignore_errors=True)
fetch_store(ABL_RUN, ABL_STORE)
ap = autopush.start(ABL_STORE, ABL_RUN, GH, minutes=10)
try:
    rc = sh(f'python -u run_ctc_validate.py --ctc {CTC} '
            f'--pool-store {TRAIN} --store-dir {ABL_STORE} --n-problems 400 '
            f'--eval-offset 1500 --flag-budget 0.40 '
            f'--type-conf-gate 1.01 --retry-flagged '          # 1.01 = ALWAYS generic
            f'--retrieval 3stage --model {MODEL} --show-every 50 '
            f'--secret-name HF_TOKEN')
finally:
    ap.stop()
assert rc==0, f'ablation failed exit {rc}'

TYPED_STORE=f'{STORES}/store_ctcval_train_{TAG}_a2'      # the stage-2 run you have
import itertools
def _fin(store):
    rs=[json.loads(l) for l in open(f'{store}/results.jsonl') if l.strip()]
    out={}
    for r in rs:
        if 'correct' not in r: continue
        out[r['id']] = bool(r['retry_correct']) if r.get('retry_trace') else bool(r['correct'])
    return out
A=_fin(TYPED_STORE); B=_fin(ABL_STORE)
ids=sorted(set(A)&set(B))
print(f'\n  paired on {len(ids)} questions   (REPLACE scoring, both arms)')
print(f'    TYPED cure    {sum(A[i] for i in ids)}/{len(ids)} = {sum(A[i] for i in ids)/len(ids):.1%}')
print(f'    GENERIC cure  {sum(B[i] for i in ids)}/{len(ids)} = {sum(B[i] for i in ids)/len(ids):.1%}')
from stats import mcnemar
mc=mcnemar(B,A)          # B = generic (baseline), A = typed
print(f"    typed-only {mc['n01']}   generic-only {mc['n10']}   McNemar p = {mc['p']:.4f}"
      + ('  SIGNIFICANT' if mc['p']<0.05 else '  NOT significant'))
print('\n  This is the cleanest number in the project: same questions, same detector,')
print('  same budget, one factor changed. Report it whichever way it lands.')